# 1.1 Build the Cohort

**Core series, class 1.** A **cohort** is a group of people chosen by a set of rules. In this class we pull our cohort out of the *All of Us* database with **SQL**: adults with health records, who either **have type 2 diabetes** (cases) or **have no diabetes** (controls). Then we take a sample and download what the rest of the series needs: their **survey answers** and their **measurements** (BMI, blood pressure, cholesterol, HbA1c).

### By the end of this notebook you'll be able to

1. Explain how the *All of Us* data is organised: the **CDR**, the **OMOP** tables, and **concept IDs**
2. Use the **concept hierarchy** (`concept_ancestor`) to find a disease and everything more specific under it
3. Read a SQL query: `SELECT`, `FROM`, `JOIN`, `WHERE`, `GROUP BY`, and `WITH`
4. Run the shared cohort query from R, and describe the result
5. Download survey answers and measurements for a sample, and save them to your bucket

### 🔎 Analysis at a glance

| | |
|---|---|
| ❓ **Question** | Who is in our type 2 diabetes cohort, and what data do we have for them? |
| 🧪 **The analysis** | Three SQL queries: the **cohort** (one row per person), **survey answers**, and **measurements**; counts and summaries |
| 🎯 **Why** | Every later class uses these files. Defining cases and controls **once**, with written rules, keeps every analysis comparable and reproducible |
| ✅ **What we conclude** | How many cases and controls there are, and how complete their data is |
| ⚠️ **What it can't tell us** | Anything about cause or risk: this class collects and describes data |
| 🔗 **Connections** | **1.2** adds diagnosis dates for the survival analysis; **2.1–2.3** clean and combine these files; the **omics** series (4.x) uses the same cohort query |

| | |
|---|---|
| 📂 **Uses** | the *All of Us* Controlled Tier dataset (BigQuery) |
| 💾 **Saves** | `cohort_sample.csv`, `survey_raw.csv`, `measurement_raw.csv` |

### Class plan

| Part | Minutes |
|---|---|
| 1. Key concepts: the CDR, OMOP, concept IDs, SQL | 20 |
| 2. Setup | 5 |
| 3. The cohort query | 25 |
| 4. Choose a sample | 10 |
| 5. Survey answers | 15 |
| 6. Measurements | 20 |
| Wrap-up | 5 |

**Before you start:**

- Run **0.0 setup**, so `aou_setup.R` has your bucket and dataset.
- This notebook is in `notebooks/core/`. It loads `../aou_setup.R` and `../sql/t2d_cohort.sql` from the folder above.

### ⏱️ Time and 💰 cost

| | |
|---|---|
| ⏱️ **Class time** | 90 minutes |
| ⏱️ **Run time** | about **10 minutes** to run every cell. Slowest: the cohort query (1–2 min) and the measurement query and download (a few min) |
| 💰 **Data cost** | **about $1 to $2**: three BigQuery queries. The measurement query reads the largest table |
| 💰 **App cost** | about **$0.27 an hour** while the app is running (the default app: 4 CPUs, 26 GB memory). The app's disk costs about **$0.70 a day**, even while the app is stopped |
| 💡 **To spend less** | run each query **once**: the files are saved to your bucket, and every later class reads them instead of querying again |
| **Estimated total** | about **$2** per student for this class |

App prices are the Workbench's estimate for the default app (October 2026). BigQuery costs are our estimates. **Stop** the app after each class.

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data, or counts of 1 to 20. Clear outputs before you share or commit this notebook (**Edit → Clear Outputs of All Cells**).

## Part 1. Key concepts

### The CDR and the OMOP tables

The *All of Us* data is stored in **BigQuery** as the **Curated Data Repository (CDR)**: a set of tables, released in numbered **versions** (we use **CDR v9**). The tables follow a standard called the **OMOP Common Data Model**, so data from many hospitals fits the same structure. The tables in this series:

| Table | One row per… | Example columns |
|---|---|---|
| `person` | participant | `person_id`, `birth_datetime`, `sex_at_birth_concept_id` |
| `condition_occurrence` | diagnosis | `person_id`, `condition_concept_id`, `condition_start_date` |
| `measurement` | lab result or vital sign | `person_id`, `measurement_concept_id`, `value_as_number`, `unit_concept_id` |
| `ds_survey` | survey answer (an easy-to-read version of the survey data) | `person_id`, `question`, `answer` |
| `visit_occurrence` | health care visit | `person_id`, `visit_start_date` |
| `death` | participant who has died | `person_id`, `death_date` |
| `concept` | code in the dictionary | `concept_id`, `concept_name` |
| `concept_ancestor` | pair of concepts where one is **under** the other | `ancestor_concept_id`, `descendant_concept_id` |

Every table shares **`person_id`**, which is how we link a person's data across tables.

### Concept IDs and the hierarchy

Instead of storing text like *"Type 2 diabetes"*, OMOP tables store a number, a **concept ID**, that points to an entry in the `concept` table. The same idea always has the same ID, whichever hospital recorded it. Look up any concept at [athena.ohdsi.org](https://athena.ohdsi.org/).

Concepts form a **hierarchy**: *Type 2 diabetes mellitus* (201826) has more specific concepts under it, like *Type 2 diabetes mellitus with diabetic nephropathy*. A doctor may record either. The `concept_ancestor` table lists every pair "this concept is under that one" (and each concept with itself), so this finds **all** of them at once:

```sql
SELECT co.person_id
FROM condition_occurrence AS co
JOIN concept_ancestor AS ca ON co.condition_concept_id = ca.descendant_concept_id
WHERE ca.ancestor_concept_id = 201826     -- type 2 diabetes, and everything under it
```

### Our cohort

| | Definition | Why |
|---|---|---|
| **Everyone** | at least one **EHR** visit (from a health care site, or records the participant shared), aged **18+** on 2025-01-01 (or at death) | without health records, "no diabetes diagnosis" only means "no records" |
| **Cases** | a **type 2 diabetes** diagnosis (201826 or anything under it), and **no** type 1 diagnosis | type 1 and type 2 have different causes; mixing them blurs every result |
| **Controls** | **no diabetes** diagnosis of any type (201820 or anything under it) | a clean comparison group |
| **Excluded** | everyone else (e.g. type 1, or both type 1 and type 2 codes) | neither clearly a case nor clearly a control |

**Age** is calculated on one fixed date, the v9 data cutoff (2025-01-01), so the numbers don't change each time the notebook runs. Race and ethnicity are **described**, not used to choose people.

The query is in one shared file, `notebooks/sql/t2d_cohort.sql`, used by **every** series, so all classes study the same people.

### SQL in one table

| SQL | Means | R equivalent |
|---|---|---|
| `SELECT a, b` | the columns to return | `select()` |
| `FROM table` | the table to read | |
| `JOIN t2 ON t1.x = t2.x` | combine rows from two tables that match | `inner_join()` |
| `LEFT JOIN` | keep every row of the first table, matched or not | `left_join()` |
| `WHERE condition` | keep only rows that meet it | `filter()` |
| `GROUP BY a` + `MIN()`, `COUNT()` | one row per group, with a summary | `group_by()` + `summarise()` |
| `IN (1, 2, 3)` | is the value in this list? | `%in%` |
| `WITH name AS (...)` | a named step, used later in the query (a **CTE**) | saving a result with `<-` |
| `-- text` | a comment | `# text` |

## Part 2. Setup

`source()` loads the shared settings and helper functions from the folder above (`..`). `run_query()`, one of the helpers, sends SQL to BigQuery and returns a data frame.

First, a check that the dataset has every table we need. `INFORMATION_SCHEMA.TABLES` is BigQuery's built-in list of the tables in a dataset. This tiny query costs nothing.

In [ ]:
source("../aou_setup.R")  # settings, packages, and helper functions

needed <- c("person", "concept", "concept_ancestor", "condition_occurrence", "visit_occurrence",
            "visit_occurrence_ext", "death", "measurement", "ds_survey")

found <- run_query(str_glue(
  "SELECT table_name FROM `{CURRENT_DATASET}`.INFORMATION_SCHEMA.TABLES"))$table_name

tibble(table = needed, found = needed %in% found)

**🔍 Read the output:** every row should say `TRUE`. If any say `FALSE`, `CURRENT_DATASET` in `aou_setup.R` probably points to the wrong dataset: it should be `wb-silky-artichoke-2408.C2025Q4R6` (CDR v9). Don't copy it from a Data Explorer query, which currently uses CDR v8.

## Part 3. The cohort query

`read_file()` reads the SQL file; `cat()` prints it, so you can read exactly what will run. **Read the comments**: every step is explained.

In [ ]:
cohort_sql <- read_file("../sql/t2d_cohort.sql")
cat(cohort_sql)

**🔍 Read the output:** the query is a chain of named steps (`WITH … AS`):

| Step | Finds |
|---|---|
| `ehr_people` | people with an EHR visit, and their first and last EHR visit dates |
| `deaths` | each person's death date, if they've died |
| `t2d`, `t1d`, `any_diabetes` | people with each kind of diabetes diagnosis (through `concept_ancestor`); for type 2, also the first date and the number of days with a code |
| `people` | demographics: sex at birth, gender, race, ethnicity, and the self-reported category, by joining `person` to `concept` to turn IDs into names |
| `people_with_age` | age on the reference date (or at death) |
| the final `SELECT` | one row per person, with the `cohort` column: **case**, **control**, or **excluded** |

The two `{…}` placeholders, `{CURRENT_DATASET}` and `{AGE_DATE}`, are filled in by `str_glue()` just before the query runs.

💰 This query costs cents to under $1, and takes a minute or two.

In [ ]:
cohort <- run_query(str_glue(cohort_sql, AGE_DATE = "2025-01-01")) %>%
  mutate(person_id = as.character(person_id))

nrow(cohort)          # adults with EHR data
cohort %>% count(cohort)

**🔍 Read the output:** over 400,000 adults with EHR data. Most are controls; about one in five is a case; a few percent are excluded.

`as.character()` stores `person_id` as **text**. IDs are labels, not numbers to do arithmetic with, and storing them as text avoids problems later (e.g. joining to files where IDs are text, or large numbers printed as `1e+06`).

### What's in the cohort table?

`glimpse()` lists every column with its type and first few values. **Don't share this output**: it shows real rows.

In [ ]:
glimpse(cohort)

**🔍 Read the output:**

| Column | Means |
|---|---|
| `person_id` | the participant |
| `date_of_birth`, `age`, `age_on` | birth date; age on `age_on` (2025-01-01, or the death date) |
| `deceased`, `death_date` | whether, and when, the person died |
| `first_ehr_date`, `last_ehr_date` | first and last EHR visit: the window in which a diagnosis **could** be recorded |
| `sex_at_birth`, `gender`, `race`, `ethnicity`, `self_reported_category` | demographics, as names |
| `first_t2d_date`, `t2d_code_days` | for cases: the first type 2 diabetes date, and how many different days had a type 2 code |
| `has_t2d`, `has_t1d`, `has_any_diabetes` | the three diabetes flags (`TRUE`/`FALSE`) |
| `cohort` | **case**, **control**, or **excluded** |

### Describe the groups

Summaries by group are safe to share (as long as no count is 1 to 20).

In [ ]:
cohort %>%
  filter(cohort != "excluded") %>%
  group_by(cohort) %>%
  summarise(people         = n(),
            mean_age       = round(mean(age), 1),
            pct_female     = round(100 * mean(sex_at_birth == "Female", na.rm = TRUE), 1),
            pct_deceased   = round(100 * mean(deceased), 1),
            median_ehr_yrs = round(median(as.numeric(last_ehr_date - first_ehr_date) / 365.25), 1))

**🔍 Read the output:** cases are **older** on average, and have **longer** EHR histories (`median_ehr_yrs`: years between their first and last EHR visit). In our test run: mean age 63 vs 55, EHR history 10.8 vs 8.3 years; cases were also less often female (56% vs 63%) and more often deceased (3.8% vs 1.3%). Both make sense: diabetes becomes more common with age, and people with a chronic condition see doctors more often.

**💬 Discuss:** people with longer health records have more chances to get **any** diagnosis recorded. How could that affect a comparison of cases and controls? (This is one form of **information bias**.)

### ✍️ Your turn

Some cases have a type 2 diabetes code on only **one** day. A single code can be a mistake, or a test to *rule out* diabetes. How many cases have 1 code day, and how many have 2 or more?

<details>
<summary><b>Show answer</b></summary>

```r
cohort %>%
  filter(cohort == "case") %>%
  count(code_days = if_else(t2d_code_days >= 2, "2 or more", "1"))
```

In our test run, about **17%** of cases had a code on only one day. Some studies (including *All of Us*'s All by All results) require **2 or more** days, which makes cases more certain but loses people. Our main definition uses 1; `t2d_code_days` lets you try the stricter one.

</details>

In [ ]:
# Your code here

## Part 4. Choose a sample

The cohort has over 400,000 people. For the core series we take a **random sample of 5,000 cases and 5,000 controls**:

- Downloading every measurement for 400,000 people would be millions of rows: slow, and more than a class needs.
- 10,000 people are plenty to learn cleaning, joining, and modelling, and to see real differences between the groups.
- BigQuery charges by the data a query **reads**, not the rows it returns, so the queries cost about the same either way.

We keep people aged **18 to 100**: a few recorded ages are above 100 (up to 119 in our test), which are probably data errors (more in **2.1 clean**).

`arrange(person_id)` then `set.seed()` makes the sample **repeatable**: BigQuery doesn't return rows in a fixed order, so we sort first; the seed then picks the same "random" people every time.

In [ ]:
N_PER_GROUP <- 5000

set.seed(2026)
cohort_sample <- cohort %>%
  filter(cohort %in% c("case", "control"), between(age, 18, 100)) %>%
  arrange(person_id) %>%
  group_by(cohort) %>%
  slice_sample(n = N_PER_GROUP) %>%
  ungroup()

cohort_sample %>% count(cohort)

save_data(cohort_sample, "cohort_sample.csv")

**🔍 Read the output:** 5,000 of each, saved to your bucket. Every later class loads `cohort_sample.csv` instead of running the query again.

**💬 Discuss:** our sample is **half cases**, but only about one in five adults in the cohort has type 2 diabetes. What does that mean for numbers like "the percent with high blood pressure" in our sample? (They describe the **sample**, not the *All of Us* population: a **case-control** sample is built to compare groups, not to estimate how common things are.)

### The IDs for the next queries

The next two queries should only return data for our 10,000 people. We put their IDs into the SQL as a list: `WHERE person_id IN (1234, 5678, …)`. `str_c(…, collapse = ", ")` joins the IDs into one text with commas.

**Don't print `ids_sql` or the queries that use it**: they contain participant IDs.

In [ ]:
ids_sql <- str_c(cohort_sample$person_id, collapse = ", ")
nchar(ids_sql)   # how long the list is, in characters

## Part 5. Survey answers

Survey answers are in `ds_survey`: one row per person per question answered, with the question and answer as text. We pull two questions from the **Personal and Family Health History** survey:

| Concept ID | Question (shortened) |
|---|---|
| 43528846 | Are you currently prescribed medications and/or receiving **treatment** for **PTSD**? |
| 43530361 | Are you still seeing a **doctor** or health care provider for **PTSD**? |

These are **follow-up** questions: participants only see them if they said they have PTSD. So most people **won't have an answer**, and that's not the same as answering "no" (**skip logic**, more in **2.1 clean** and the **survey series**).

💰 Cents.

In [ ]:
survey_sql <- str_glue("
SELECT person_id, question_concept_id, question, answer, survey_datetime
FROM `{CURRENT_DATASET}.ds_survey`
WHERE question_concept_id IN (43528846, 43530361)
  AND person_id IN ({ids_sql})
")

survey_raw <- run_query(survey_sql) %>% mutate(person_id = as.character(person_id))

nrow(survey_raw)                   # answers
n_distinct(survey_raw$person_id)   # people who answered at least one

# The answers given (labels and counts, not people)
survey_raw %>% count(question_concept_id, answer) %>% hide_small_counts()

save_data(survey_raw, "survey_raw.csv")

**🔍 Read the output:**

- Only a **small share** of our 10,000 people answered: the questions were only asked of people with PTSD.
- Each answer **repeats the question**, then the answer: *"Are you still seeing a doctor … (PTSD)? - Yes"*. 2.1 keeps just the part after " - ".
- There are also **non-answers**: in our test run, *PMI: Skip* (the participant skipped the question). Other surveys also use *PMI: Prefer Not To Answer* and *PMI: Dont Know*. Cleaning these is part of **2.1**.
- In our test run, about 500 of our 10,000 people answered each question.
- Some people may have **more than one** answer to the same question (e.g. they took the survey twice): another job for 2.1.

## Part 6. Measurements

Measurements are in `measurement`: one row per test result, with `value_as_number` and a **unit** (`unit_concept_id`). We ask for four measures, each identified by one or more concept IDs (checked in the OHDSI vocabulary):

| Measure | Concept IDs | Usual unit |
|---|---|---|
| **BMI** (body mass index) | 3038553 | kg/m² |
| **Systolic blood pressure** | 3004249, 4152194 | mm Hg |
| **HbA1c** (average blood sugar over ~3 months) | 3004410 | % |
| **LDL cholesterol** | 3028437, 3028288, 3009966, 36031404 | mg/dL |

The query **joins** `concept` twice: once to turn the measurement's concept ID into its name, and once (a `LEFT JOIN`, because some results have no unit) to turn the unit's concept ID into its name. `AS m`, `AS c`, `AS u` give each table a short **alias**.

💰 About $1: `measurement` is one of the biggest tables. It takes a few minutes, including the download.

In [ ]:
measurement_sql <- str_glue("
SELECT m.person_id,
       m.measurement_concept_id,
       c.concept_name AS measurement,
       m.measurement_date,
       m.value_as_number,
       u.concept_name AS unit
FROM `{CURRENT_DATASET}.measurement` AS m
JOIN `{CURRENT_DATASET}.concept` AS c ON m.measurement_concept_id = c.concept_id
LEFT JOIN `{CURRENT_DATASET}.concept` AS u ON m.unit_concept_id = u.concept_id
WHERE m.measurement_concept_id IN (3038553, 3004249, 4152194, 3004410, 3028437, 3028288, 3009966, 36031404)
  AND m.person_id IN ({ids_sql})
")

measurement_raw <- run_query(measurement_sql) %>% mutate(person_id = as.character(person_id))

nrow(measurement_raw)   # measurements: many per person

# Rows and people per measurement concept
measurement_raw %>%
  group_by(measurement_concept_id, measurement) %>%
  summarise(rows = n(), people = n_distinct(person_id), .groups = "drop") %>%
  arrange(desc(rows))

**🔍 Read the output:**

- **Many rows per person**: people are measured at every visit. In our test run, systolic blood pressure had **over a million rows** for about 9,800 people. In **2.2 reshape** we pick one value per person per measure.
- Concept 4152194 (also systolic blood pressure) had about **200 rows per person** for only ~1,400 people: likely repeated readings during **hospital stays**. Worth remembering when choosing "one value per person".
- Some concept IDs have far more rows than others: blood pressure and BMI are measured at most visits; HbA1c and LDL only when ordered.
- The `measurement` names show why we needed **several** IDs for systolic blood pressure and LDL: different labs record them under different (but equivalent) concepts.

### Units

The same measure can be recorded in different **units**, or with no unit at all. Before using any number, check its unit:

In [ ]:
measurement_raw %>%
  count(measurement, unit, sort = TRUE) %>%
  hide_small_counts()

save_data(measurement_raw, "measurement_raw.csv")
list.files(DATA_DIR)

**🔍 Read the output:**

- Many results use the expected unit (*kilogram per square meter*, *millimeter mercury column*, *percent*, *milligram per deciliter*), sometimes written differently (*kg/sq. m*, *mg/dL*).
- A lot say **"No matching concept"**: the site's unit couldn't be mapped to a standard one. In our test run that was almost **half** of all blood pressure rows. Unmapped doesn't mean wrong: the values are usually fine.
- Some units are clearly **mapping errors**: in our test run, BMI with the unit *"link for Gunter's chain (US)"* (a surveying length!), or systolic blood pressure with the unit *"Systolic blood pressure"*.
- Some **look** like a different measure: HbA1c in *gram per deciliter* or *milligram per deciliter* can't be an HbA1c percentage. Others only look wrong: BMI in *percent* might be a child's BMI percentile, but in our test run the values were ordinary adult BMIs. **2.1** checks the values before deciding.
- **The lesson:** a unit label alone can't be trusted. **2.1 clean** combines the unit with a **plausible range** of values for each measure, and looks at the values unit by unit before deciding what to keep.
- The list of files shows the three you saved: `cohort_sample.csv`, `survey_raw.csv`, `measurement_raw.csv`.

### ✍️ Your turn

What range of values does **BMI** have? Use `summary()` on BMI's `value_as_number`. Do any values look impossible?

<details>
<summary><b>Show answer</b></summary>

```r
measurement_raw %>%
  filter(measurement_concept_id == 3038553) %>%
  pull(value_as_number) %>%
  summary()
```

Most BMIs fall between about 18 and 45, but the minimum and maximum are often **impossible** (e.g. 0, or several hundred): typing errors, or a weight or height entered in the wrong box. `NA's` counts results with no number. **2.1 clean** sets rules for these.

</details>

In [ ]:
# Your code here

## Prefer point and click?

The **Data Explorer** (Workbench 2.0's point-and-click cohort tool) can build a similar cohort without SQL. But it currently uses **CDR v8**, not v9, and participant IDs changed between the two, so a Data Explorer cohort **can't be combined** with this series' files. Its rules also differ a little (e.g. age **today** instead of on 2025-01-01), so its counts won't match. For teaching, use one method consistently; this series uses the SQL, so every class studies exactly the same people.

## ⚠️ Common errors

| Message | Means | Fix |
|---|---|---|
| *cannot open file '../aou_setup.R'* | The notebook isn't in `notebooks/core/` | Keep the repository's folder layout |
| *Access Denied* / *Not found: Dataset* | Wrong `CURRENT_DATASET`, or no Controlled Tier access | Check `aou_setup.R` says `wb-silky-artichoke-2408.C2025Q4R6` (CDR v9) |
| *Not found: Table … ds_survey* | This dataset doesn't have the survey table | Check Part 2's table list |
| *Query exceeds … maximum length* | Too many IDs in `IN (…)` | Keep `N_PER_GROUP` at 5,000 or below |
| *object 'cohort_sample' not found* | The sample cell didn't run (e.g. after a restart) | `cohort_sample <- load_data("cohort_sample.csv", col_types = cols(person_id = "c"))` |

## ✅ Key takeaways

- *All of Us* data is in **OMOP** tables, linked by `person_id`; codes are **concept IDs**
- `concept_ancestor` finds a concept **and everything under it**
- One **shared query** defines the cohort for every class
- Query **once**, save to your bucket, and load the files afterwards
- **Check units** and look for impossible values before using a measurement

**Next:** **1.2 survival_data**: the dates needed to study **time to chronic kidney disease** after a type 2 diabetes diagnosis.

**Before you share or commit this notebook:** clear all outputs (**Edit → Clear Outputs of All Cells**).